# Day 2 — Silver Layer: Clean, Cast, Conform
### PySpark transformations · String cleaning · Type casting · Window functions · SQL Views
### Retail Analytics DE Project

---

> **Project Day:** 2 · **Layer:** Silver  
> **Input:** `bronze.*` tables in PostgreSQL  
> **Output:** `silver.*` tables in PostgreSQL  
> **Prerequisite:** Day 1 complete — bronze tables populated

| Section | Topic |
|---------|-------|
| 1 | Setup — import config + Spark |
| 2 | Silver — Customers |
| 3 | Silver — Orders (value tier + lookup map) |
| 4 | Silver — Order Items (line_total + rank window) |
| 5 | Silver — Products + Inventory |
| 6 | Silver — Store Locations |
| 7 | Silver — Web Logs |
| 8 | Silver — Weather |
| 9 | SQL Views on Silver |
| 10 | Silver Summary |

---
## 1. Setup — Import config + Spark

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from config.db_config import (
    get_engine, set_spark_env, get_spark,
    spark_read_pg, spark_write_pg
)

engine = get_engine()
print('Engine ready:', engine.url)

In [ ]:
# Must set env BEFORE any pyspark import
set_spark_env()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, BooleanType

spark = get_spark('Day2_Silver')
print('Spark ready:', spark.version)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from config.db_config import spark_read_pg, spark_write_pg, JDBC_URL, DB_USER, DB_PASS
from sqlalchemy import text, inspect as sa_inspect
from datetime import datetime

SILVER_AT = datetime.utcnow().isoformat()

def read_bronze(table):
    """Read a bronze table from PostgreSQL into a PySpark DataFrame via JDBC."""
    df = spark_read_pg(spark, 'bronze', table)
    print(f'  bronze.{table}: {df.count()} rows')
    return df

def drop_meta_cols(df):
    """Drop bronze metadata columns before writing to silver."""
    meta = ['_source_file', '_ingested_at', '_batch_id', '_api_source', '_api_fetched_at']
    return df.drop(*[c for c in meta if c in df.columns])

def to_silver(df, table):
    """Add _silver_loaded_at and persist to silver schema via JDBC."""
    df = df.withColumn('_silver_loaded_at', F.lit(SILVER_AT))
    spark_write_pg(df, 'silver', table, mode='overwrite')

print('Helpers ready. SILVER_AT =', SILVER_AT)

---
---
## 2. Silver — Customers

---

| Column | Raw | Silver |
|--------|-----|--------|
| email | `' Alice@Gmail.COM '` | `'alice@gmail.com'` |
| city | `'new york'` | `'New York'` |
| state | `'ny'` | `'NY'` |
| signup_date | `'2022-01-15'` (string) | DateType |
| is_active | `'true'` (string) | `True` (Boolean) |
| full_name | — | `first_name + ' ' + last_name` |

In [ ]:
df_cust = read_bronze('customers')
print('Bronze schema:')
df_cust.printSchema()

In [ ]:
df_cust_s = (
    df_cust
    # String cleaning
    .withColumn('first_name', F.trim(F.col('first_name')))
    .withColumn('last_name',  F.trim(F.col('last_name')))
    .withColumn('email',      F.lower(F.trim(F.col('email'))))
    .withColumn('city',       F.initcap(F.trim(F.col('city'))))
    .withColumn('state',      F.upper(F.trim(F.col('state'))))
    # Derived
    .withColumn('full_name', F.concat_ws(' ', F.col('first_name'), F.col('last_name')))
    # Type casting
    .withColumn('signup_date', F.to_date(F.col('signup_date'), 'yyyy-MM-dd'))
    .withColumn('is_active',
        F.when(F.lower(F.col('is_active')).isin('true','1','yes'), F.lit(True))
         .otherwise(F.lit(False)).cast(BooleanType())
    )
    .dropDuplicates(['customer_id'])
)
df_cust_s = drop_meta_cols(df_cust_s)
df_cust_s.select('customer_id','full_name','email','signup_date','is_active','city','state').show(5, truncate=False)
to_silver(df_cust_s, 'customers')

---
---
## 3. Silver — Orders

---

**Derived columns:**
- `is_cancelled` — boolean from status string
- `order_value_tier` — `CASE WHEN` on total_amount
- `payment_method_clean` — lookup join using `create_map()` (PySpark's dict-to-column trick)

In [ ]:
df_ord = read_bronze('orders')

# Build a MapType column from a Python dict for the payment lookup
PAYMENT_MAP = {
    'credit_card'  : 'Credit Card',
    'debit_card'   : 'Debit Card',
    'paypal'       : 'PayPal',
    'bank_transfer': 'Bank Transfer',
    'cash'         : 'Cash',
}
# create_map expects alternating key, value literals
map_col = F.create_map(*[F.lit(x) for pair in PAYMENT_MAP.items() for x in pair])

df_ord_s = (
    df_ord
    .withColumn('total_amount', F.col('total_amount').cast(DoubleType()))
    .withColumn('order_date',   F.to_date(F.col('order_date'), 'yyyy-MM-dd'))
    .withColumn('status',       F.lower(F.trim(F.col('status'))))
    # Derived: boolean flag
    .withColumn('is_cancelled',
        F.when(F.col('status') == 'cancelled', F.lit(True))
         .otherwise(F.lit(False)).cast(BooleanType())
    )
    # Derived: value tier
    .withColumn('order_value_tier',
        F.when(F.col('total_amount') >= 200, F.lit('high'))
         .when(F.col('total_amount') >= 100, F.lit('medium'))
         .otherwise(F.lit('low'))
    )
    # Lookup join using MapType
    .withColumn('payment_method_clean',
        F.coalesce(map_col[F.col('payment_method')], F.col('payment_method'))
    )
    .dropDuplicates(['order_id'])
)

df_ord_s = drop_meta_cols(df_ord_s)
print('Value tier distribution:')
df_ord_s.groupBy('order_value_tier').count().orderBy('order_value_tier').show()
to_silver(df_ord_s, 'orders')

---
---
## 4. Silver — Order Items (line_total + rank window)

---

**Window functions:**

| Function | Window | Result |
|----------|--------|--------|
| `RANK()` | `PARTITION BY order_id ORDER BY line_total DESC` | Rank of each item by value within its order |
| `SUM() running` | `PARTITION BY order_id ORDER BY item_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` | Running total of revenue per order |

In [ ]:
df_items = read_bronze('order_items')

df_items_s = (
    df_items
    .withColumn('quantity',     F.col('quantity').cast(IntegerType()))
    .withColumn('unit_price',   F.col('unit_price').cast(DoubleType()))
    .withColumn('discount_pct', F.col('discount_pct').cast(DoubleType()))
    .fillna({'discount_pct': 0.0})
    # Derived: net revenue for this line
    .withColumn(
        'line_total',
        F.round(F.col('quantity') * F.col('unit_price') * (1 - F.col('discount_pct') / 100), 2)
    )
)

# Window 1: rank items within each order by value descending
w_rank = Window.partitionBy('order_id').orderBy(F.col('line_total').desc())
df_items_s = df_items_s.withColumn('rank_in_order', F.rank().over(w_rank))

# Window 2: running total within each order
w_run = Window.partitionBy('order_id').orderBy('item_id').rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)
df_items_s = df_items_s.withColumn(
    'running_order_total', F.round(F.sum('line_total').over(w_run), 2)
)

df_items_s = drop_meta_cols(df_items_s).dropDuplicates(['item_id'])

print('Order O001 — items ranked:')
df_items_s.filter(F.col('order_id') == 'O001') \
    .select('item_id','order_id','product_id','quantity','unit_price','discount_pct','line_total','rank_in_order','running_order_total') \
    .orderBy('rank_in_order') \
    .show(truncate=False)

to_silver(df_items_s, 'order_items')

---
---
## 5. Silver — Products + Inventory

---

In [ ]:
df_prod = read_bronze('products')

df_prod_s = (
    df_prod
    .withColumn('product_name', F.trim(F.col('product_name')))
    .withColumn('brand',        F.trim(F.col('brand')))
    .withColumn('category',     F.upper(F.trim(F.col('category'))))
    .withColumn('unit_price',   F.col('unit_price').cast(DoubleType()))
    .withColumn('is_available',
        F.when(F.lower(F.col('is_available')).isin('true','1'), F.lit(True))
         .otherwise(F.lit(False)).cast(BooleanType())
    )
    .dropDuplicates(['product_id'])
)
df_prod_s = drop_meta_cols(df_prod_s)
print('Unique categories:')
df_prod_s.select('category').distinct().orderBy('category').show(truncate=False)
to_silver(df_prod_s, 'products')

In [ ]:
df_inv = read_bronze('inventory')

df_inv_s = (
    df_inv
    .withColumn('stock_qty',    F.col('stock_qty').cast(IntegerType()))
    .withColumn('reorder_level',F.col('reorder_level').cast(IntegerType()))
    .withColumn('last_updated', F.to_date(F.col('last_updated')))
    # Derived: stock below reorder threshold
    .withColumn('is_low_stock',
        F.when(F.col('stock_qty') <= F.col('reorder_level'), F.lit(True))
         .otherwise(F.lit(False)).cast(BooleanType())
    )
    # Derived: days since last stock update
    .withColumn('days_since_update', F.datediff(F.current_date(), F.col('last_updated')))
    .dropDuplicates(['product_id', 'warehouse_id'])
)
df_inv_s = drop_meta_cols(df_inv_s)
print('Low stock items:')
df_inv_s.filter(F.col('is_low_stock') == True) \
    .select('product_id','warehouse_id','stock_qty','reorder_level','is_low_stock') \
    .show(truncate=False)
to_silver(df_inv_s, 'inventory')

---
---
## 6. Silver — Store Locations

---

In [ ]:
df_stores = read_bronze('store_locations')

df_stores_s = (
    df_stores
    .withColumn('store_name',  F.trim(F.col('store_name')))
    .withColumn('city',        F.initcap(F.trim(F.col('city'))))
    .withColumn('state',       F.upper(F.trim(F.col('state'))))
    .withColumn('lat',         F.col('lat').cast(DoubleType()))
    .withColumn('lon',         F.col('lon').cast(DoubleType()))
    .withColumn('opened_date', F.to_date(F.col('opened_date')))
    .dropDuplicates(['store_id'])
)
df_stores_s = drop_meta_cols(df_stores_s)
df_stores_s.show(truncate=False)
to_silver(df_stores_s, 'store_locations')

---
---
## 7. Silver — Web Logs

---

Add derived columns: `is_error` (status ≥ 400), `response_kb`.

In [ ]:
df_logs = read_bronze('web_logs')

df_logs_s = (
    df_logs
    .withColumn('status_code',   F.col('status_code').cast(IntegerType()))
    .withColumn('response_size', F.col('response_size').cast(DoubleType()))
    .withColumn('endpoint',      F.trim(F.col('endpoint')))
    .withColumn('method',        F.upper(F.trim(F.col('method'))))
    .withColumn('is_error',
        F.when(F.col('status_code') >= 400, F.lit(True))
         .otherwise(F.lit(False)).cast(BooleanType())
    )
    .withColumn('response_kb', F.round(F.col('response_size') / 1024, 2))
)
df_logs_s = drop_meta_cols(df_logs_s)

print('Method distribution:')
df_logs_s.groupBy('method').count().show()
print('Error rate:')
df_logs_s.groupBy('is_error').count().show()
to_silver(df_logs_s, 'web_logs')

---
---
## 8. Silver — Weather (file + live API)

---

In [ ]:
# Weather from static JSON file (ingested Day 1)
df_wf = read_bronze('weather_file')
df_wf_s = (
    df_wf
    .withColumn('temp_c',       F.col('temp_c').cast(DoubleType()))
    .withColumn('humidity_pct', F.col('humidity_pct').cast(DoubleType()))
    .withColumn('wind_kmh',     F.col('wind_kmh').cast(DoubleType()))
    .withColumn('city',         F.initcap(F.trim(F.col('city'))))
    .dropDuplicates(['city'])
)
df_wf_s = drop_meta_cols(df_wf_s)
to_silver(df_wf_s, 'weather_file')

# Weather from live Open-Meteo API call (ingested Day 1)
df_wl = read_bronze('weather_live')
df_wl_s = (
    df_wl
    .withColumn('temp_c',       F.col('temp_c').cast(DoubleType()))
    .withColumn('humidity_pct', F.col('humidity_pct').cast(DoubleType()))
    .withColumn('wind_kmh',     F.col('wind_kmh').cast(DoubleType()))
    .withColumn('weather_code', F.col('weather_code').cast(IntegerType()))
    .withColumn('lat',          F.col('lat').cast(DoubleType()))
    .withColumn('lon',          F.col('lon').cast(DoubleType()))
    .withColumn('city',         F.initcap(F.trim(F.col('city'))))
    .dropDuplicates(['city'])
)
df_wl_s = drop_meta_cols(df_wl_s)
df_wl_s.show(truncate=False)
to_silver(df_wl_s, 'weather_live')

---
---
## 9. SQL Views on Silver

---

Views are saved queries — BI tools can query them exactly like tables without knowing the joins.

In [ ]:
from sqlalchemy import text

VIEWS = [
    ('v_order_summary', """
        CREATE OR REPLACE VIEW silver.v_order_summary AS
        SELECT
            o.order_id, o.order_date, o.status,
            o.total_amount, o.order_value_tier, o.is_cancelled,
            c.full_name AS customer_name,
            c.city AS customer_city, c.state AS customer_state,
            COUNT(oi.item_id) AS item_count
        FROM silver.orders o
        LEFT JOIN silver.customers  c  ON o.customer_id = c.customer_id
        LEFT JOIN silver.order_items oi ON o.order_id  = oi.order_id
        GROUP BY o.order_id, o.order_date, o.status, o.total_amount,
                 o.order_value_tier, o.is_cancelled,
                 c.full_name, c.city, c.state
    """),
    ('v_product_stock', """
        CREATE OR REPLACE VIEW silver.v_product_stock AS
        SELECT
            p.product_id, p.product_name, p.category, p.brand, p.unit_price,
            i.warehouse_id, i.stock_qty, i.reorder_level,
            i.is_low_stock, i.days_since_update
        FROM silver.products p
        LEFT JOIN silver.inventory i ON p.product_id = i.product_id
    """),
    ('v_top_endpoints', """
        CREATE OR REPLACE VIEW silver.v_top_endpoints AS
        SELECT endpoint, COUNT(*) AS hit_count,
               SUM(CASE WHEN is_error THEN 1 ELSE 0 END) AS error_count
        FROM silver.web_logs
        GROUP BY endpoint
        ORDER BY hit_count DESC
        LIMIT 10
    """),
]

engine = get_engine()
with engine.connect() as conn:
    for name, sql in VIEWS:
        conn.execute(text(sql))
        print(f'  Created: silver.{name}')
    conn.commit()

# Read the view via PySpark JDBC
print('\nv_order_summary sample:')
df_v = spark_read_pg(spark, 'silver', 'v_order_summary')
df_v.select('order_id','customer_name','order_date','total_amount','order_value_tier','item_count').show(5, truncate=False)

---
---
## 10. Silver Summary

---

In [ ]:
from config.db_config import get_engine
from sqlalchemy import inspect as sa_inspect

engine = get_engine()
insp = sa_inspect(engine)
tables = sorted(insp.get_table_names(schema='silver'))

print(f'{"Table":<35} {"Rows":>6}')
print('-' * 43)
for t in tables:
    with engine.connect() as conn:
        from sqlalchemy import text
        n = conn.execute(text(f'SELECT COUNT(*) FROM silver.{t}')).scalar()
    print(f'  silver.{t:<28} {n:>6}')

print('\nDay 2 complete — Silver layer ready!')

In [ ]:
spark.stop()
print('SparkSession stopped.')